# Evaluation experiments — uzbek-gpt-103m

Two controlled experiments validating the paper's claims, run on a single RTX 4090.

- **Experiment A (Cell: QLoRA budget sweep):** fine-tunes mGPT-1.3B with QLoRA on increasing amounts of Uzbek (0 / 1M / 10M tokens) and scores each in bits-per-byte. Shows the adapted baseline plateaus at ~1.120, above the from-scratch model's 1.105 — more data does not close the gap.
- **Experiment B (Cell: tokenizer ablation):** trains the *same* 103M architecture from scratch on the *same* text but with mGPT's tokenizer (a larger, 232M model). It scores 1.158 bpb — worse — so the tokenizer, not training from scratch, is the cause of the advantage.

**Setup:** requires a Kaggle API token (entered via `getpass`, not stored) to download the tokenized dataset `islombekturdiyev/uzbek-fineweb2-tokens-16k`, and the published model/tokenizer from Hugging Face. Enable a GPU before running.


In [ ]:
# ===== CELL 0 — setup: deps + your tokenized data from Kaggle =====
!pip install -q --upgrade kaggle transformers datasets peft bitsandbytes accelerate safetensors tokenizers huggingface_hub matplotlib

import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste Kaggle token (KGAT_...) then Enter: ')
!kaggle datasets download -d islombekturdiyev/uzbek-fineweb2-tokens-16k -p /workspace --unzip

for f in ['/workspace/train.bin', '/workspace/val.bin']:
    print(f, '✓' if os.path.exists(f) else '✗ MISSING',
          round(os.path.getsize(f)/1e6) if os.path.exists(f) else '', 'MB')

In [ ]:
# ===== EXPERIMENT A — QLoRA budget sweep: 1M -> 10M -> 50M -> 100M tokens =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc, sys, math, json, numpy as np, torch
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, default_data_collator)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

BASE       = "ai-forever/mGPT"
CTX_TRAIN  = 512                       # 4090 handles 512 fine; better adaptation than 256
PER_DEV    = 8                         # 4090 has headroom; raise to 12-16 if no OOM
BUDGETS    = [1_000_000, 10_000_000, 50_000_000, 100_000_000]   # tokens actually trained on
POOL_TOKENS = 120_000_000              # cache a bit more than the largest budget
EVAL_WORDS = 200_000
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DT   = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16: {BF16}")
def free(): gc.collect(); torch.cuda.empty_cache()

# ---- frozen held-out eval set (reuse if present so it matches your existing Table 2) ----
EVAL_CACHE = "uz_heldout.txt"
if os.path.exists(EVAL_CACHE):
    EVAL_TEXT = open(EVAL_CACHE, encoding="utf-8").read(); print("Reusing", EVAL_CACHE)
else:
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    docs, w = [], 0
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        docs.append(t); w += len(t.split())
        if w >= EVAL_WORDS: break
    EVAL_TEXT = "\n".join(docs); open(EVAL_CACHE, "w", encoding="utf-8").write(EVAL_TEXT)
    print("Froze eval set ->", EVAL_CACHE)
EVAL_BYTES = len(EVAL_TEXT.encode("utf-8"))
print(f"Eval set: {len(EVAL_TEXT.split()):,} words, {EVAL_BYTES:,} bytes\n")

tok_m = AutoTokenizer.from_pretrained(BASE)
if tok_m.pad_token is None: tok_m.pad_token = tok_m.eos_token
tok_m.model_max_length = 10**9

@torch.no_grad()
def bpb_and_ce(model, ctx=512):
    ids = torch.tensor(tok_m(EVAL_TEXT, add_special_tokens=False)["input_ids"], dtype=torch.long)
    n = len(ids)//ctx; C = ids[:n*ctx].view(n, ctx); dev = next(model.parameters()).device
    nll, tok = 0.0, 0
    for i in range(0, len(C), 4):
        x = C[i:i+4].to(dev); out = model(x)
        lg = out[0] if isinstance(out, tuple) else getattr(out, "logits", out)
        nll += torch.nn.functional.cross_entropy(
            lg[:, :-1, :].float().reshape(-1, lg.size(-1)), x[:, 1:].reshape(-1),
            reduction="sum").item()
        tok += x[:, 1:].numel()
    return (nll/math.log(2))/EVAL_BYTES, nll/tok

# ---- tokenize a shared pool once ----
POOL = f"mgpt_pool_{POOL_TOKENS}.npy"
if os.path.exists(POOL):
    data = np.load(POOL); print(f"Loaded pool: {len(data)/1e6:.0f}M tokens")
else:
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    ids = []
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        ids.extend(tok_m(t, add_special_tokens=False)["input_ids"] + [tok_m.eos_token_id])
        if len(ids) >= POOL_TOKENS: break
    data = np.asarray(ids[:POOL_TOKENS], dtype=np.int32); np.save(POOL, data)
    print(f"Tokenized pool: {len(data)/1e6:.0f}M tokens")

class Blocks(Dataset):
    def __init__(s, a, c, n_seq): s.a, s.c, s.n = a, c, n_seq
    def __len__(s): return s.n
    def __getitem__(s, i):
        x = s.a[i*s.c:(i+1)*s.c].astype(np.int64)
        return {"input_ids": torch.from_numpy(x), "labels": torch.from_numpy(x.copy())}

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)

results = []
# baseline (0 adaptation) for the curve's left anchor
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"":0}).eval()
b0, c0 = bpb_and_ce(base); print(f"[budget      0] bpb={b0:.4f} ce={c0:.3f}")
results.append({"tokens":0, "bpb":b0, "ce":c0}); del base; free()

for budget in BUDGETS:
    steps = budget // (PER_DEV * CTX_TRAIN)
    n_seq = budget // CTX_TRAIN
    print(f"\n===== budget {budget/1e6:.0f}M tokens  ->  {steps} steps =====")
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"":0})
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                     task_type="CAUSAL_LM", target_modules="all-linear"))
    args = TrainingArguments(
        output_dir=f"adapter_{budget}", per_device_train_batch_size=PER_DEV,
        gradient_accumulation_steps=1, max_steps=steps, learning_rate=2e-4,
        lr_scheduler_type="cosine", warmup_steps=max(10, steps//30),
        fp16=not BF16, bf16=BF16, gradient_checkpointing=True, optim="paged_adamw_8bit",
        logging_steps=max(10, steps//20), save_strategy="no",
        dataloader_num_workers=2, report_to="none")
    Trainer(model=m, args=args, train_dataset=Blocks(data, CTX_TRAIN, n_seq),
            data_collator=default_data_collator).train()
    m.eval()
    b, c = bpb_and_ce(m)
    print(f"[budget {budget/1e6:>5.0f}M] bpb={b:.4f} ce={c:.3f}")
    results.append({"tokens":budget, "bpb":b, "ce":c})
    m.save_pretrained(f"adapter_{budget}")     # saved to /workspace, download after
    del m; free()

json.dump(results, open("qlora_sweep.json","w"), indent=2)
print("\n=== QLoRA budget sweep ===")
print(f"{'tokens':>12} {'bpb':>8} {'ce(nats)':>10}")
for r in results: print(f"{r['tokens']:>12,} {r['bpb']:>8.4f} {r['ce']:>10.3f}")
print("YOUR from-scratch model bpb = 1.1050 (reference line)")
print("Saved -> qlora_sweep.json  (send me this file)")

In [ ]:
# ===== plot the curve (send me qlora_curve.png too) =====
import json, matplotlib.pyplot as plt
r = json.load(open("qlora_sweep.json"))
x = [d["tokens"]/1e6 for d in r]; y = [d["bpb"] for d in r]
plt.figure(figsize=(7,4.5))
plt.plot(x, y, "o-", label="mGPT-1.3B + QLoRA")
plt.axhline(1.1050, ls="--", color="crimson", label="uzbek-gpt-103m (from scratch)")
plt.xscale("symlog"); plt.xlabel("QLoRA adaptation tokens (millions)")
plt.ylabel("bits-per-byte (lower = better)")
plt.title("Does more adaptation data close the gap?"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("qlora_curve.png", dpi=150); plt.show()

In [ ]:
import gc, torch
for v in ['m','base','model','abl']:
    if v in dir(): 
        try: del globals()[v]
        except: pass
gc.collect(); torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# ===== EXPERIMENT B — tokenizer ablation: YOUR 103M architecture, mGPT's tokenizer, same corpus text =====
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, math, time, json, numpy as np, torch, torch.nn.functional as F
from contextlib import nullcontext
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

# ---- pull YOUR model.py so the architecture is identical to the original run ----
mp = hf_hub_download("IslombekT/uzbek-gpt-103m", "model.py")
sys.path.insert(0, os.path.dirname(mp))
from model import GPT

BASE_TOK   = "ai-forever/mGPT"
CORPUS_WORDS = 955_000_000        # same corpus coverage as your original train split
EPOCHS     = 2                    # same ~2 passes as your original run
# --- your original body hyperparameters, unchanged ---
n_embd, n_layer, n_head, block_size = 768, 12, 12, 1024
micro_batch, grad_accum = 8, 24   # effective batch 192, as before
peak_lr, min_lr, weight_decay, grad_clip = 3e-4, 3e-5, 0.1, 1.0
warmup_steps, eval_every, eval_iters = 400, 500, 40

BF16 = torch.cuda.is_bf16_supported()
DT = torch.bfloat16 if BF16 else torch.float16
ctx = torch.autocast("cuda", dtype=torch.bfloat16) if BF16 else nullcontext()
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16: {BF16}")

tok = AutoTokenizer.from_pretrained(BASE_TOK)
VOCAB = tok.vocab_size
EOT = tok.eos_token_id if tok.eos_token_id is not None else 0
print(f"mGPT tokenizer vocab = {VOCAB:,}  (yours was 16,384)")

# ---- re-tokenize the SAME corpus text with mGPT's tokenizer -> train.bin_mgpt ----
BIN = "train_mgpt.bin"
if os.path.exists(BIN):
    train_data = np.memmap(BIN, dtype=np.int32, mode="r")
    print(f"Reusing {BIN}: {len(train_data)/1e6:.0f}M mGPT-tokens")
else:
    print("Re-tokenizing corpus with mGPT tokenizer (this streams FineWeb-2)...")
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    ids, words = [], 0
    t0 = time.time()
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        ids.extend(tok(t, add_special_tokens=False)["input_ids"] + [EOT])
        words += len(t.split())
        if words % 5_000_000 < 1000:
            print(f"  {words/1e6:.0f}M words -> {len(ids)/1e6:.0f}M tokens  ({(time.time()-t0)/60:.0f} min)")
        if words >= CORPUS_WORDS: break
    train_data = np.asarray(ids, dtype=np.int32)
    train_data.tofile(BIN)
    print(f"Wrote {BIN}: {len(train_data)/1e6:.0f}M mGPT-tokens from {words/1e6:.0f}M words")
    train_data = np.memmap(BIN, dtype=np.int32, mode="r")

TOTAL_TOKENS = len(train_data)
tokens_per_step = micro_batch * grad_accum * block_size
max_steps = (TOTAL_TOKENS * EPOCHS) // tokens_per_step
print(f"tokens/step={tokens_per_step:,}  total={TOTAL_TOKENS/1e6:.0f}M  epochs={EPOCHS}  -> {max_steps} steps\n")

def get_batch(bs, bl):
    ix = torch.randint(len(train_data) - bl - 1, (bs,))
    x = torch.stack([torch.from_numpy(train_data[i:i+bl].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(train_data[i+1:i+bl+1].astype(np.int64)) for i in ix])
    return x.cuda(), y.cuda()

def get_lr(step):
    if step < warmup_steps: return peak_lr * (step+1) / warmup_steps
    if step >= max_steps: return min_lr
    p = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5*(1+math.cos(math.pi*p))*(peak_lr - min_lr)

torch.manual_seed(1337); torch.set_float32_matmul_precision("high")
model = GPT(VOCAB, n_embd, block_size, n_head, n_layer).cuda()
n_params = sum(p.numel() for p in model.parameters())/1e6
n_embed_params = VOCAB * n_embd / 1e6
print(f"ablation model: {n_params:.1f}M params ({n_embed_params:.1f}M in embeddings alone)")
print(f"  vs your original: 103.1M params (0.25M embeddings). Same 12L/768d body.\n")
model = torch.compile(model)
opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, betas=(0.9,0.95), weight_decay=weight_decay)

@torch.no_grad()
def est_val():
    model.eval(); L=[]
    for _ in range(eval_iters):
        xb, yb = get_batch(micro_batch, block_size)
        with ctx:
            lo = model(xb); B,T,V = lo.shape
            L.append(F.cross_entropy(lo.view(B*T,V), yb.view(B*T)).item())
    model.train(); return sum(L)/len(L)

best = float("inf"); run0 = time.time(); t0 = time.time()
model.train()
for step in range(max_steps):
    for g in opt.param_groups: g["lr"] = get_lr(step)
    opt.zero_grad(set_to_none=True)
    for _ in range(grad_accum):
        xb, yb = get_batch(micro_batch, block_size)
        with ctx:
            lo = model(xb); B,T,V = lo.shape
            loss = F.cross_entropy(lo.view(B*T,V), yb.view(B*T)) / grad_accum
        loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    opt.step()
    if step % eval_every == 0 or step == max_steps-1:
        v = est_val(); dt = time.time()-t0; t0 = time.time()
        print(f"step {step:5d}/{max_steps} | val {v:.4f} | {dt:.0f}s | {(time.time()-run0)/3600:.2f}h")
        cfg = dict(vocab_size=VOCAB, n_embd=n_embd, n_layer=n_layer, n_head=n_head, block_size=block_size)
        raw = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save({"model": raw.state_dict(), "step": step, "val": v, "config": cfg}, "ablation_last.pt")
        if v < best:
            best = v
            torch.save({"model": raw.state_dict(), "step": step, "val": v, "config": cfg}, "ablation_best.pt")
print(f"DONE | best val {best:.4f} | {(time.time()-run0)/3600:.2f}h")

In [ ]:
# ===== score the ablation on the SAME frozen held-out set, in bits-per-byte =====
import math, numpy as np, torch
from transformers import AutoTokenizer

TEXT = open("uz_heldout.txt", encoding="utf-8").read()
BYTES = len(TEXT.encode("utf-8"))
tok = AutoTokenizer.from_pretrained("ai-forever/mGPT"); tok.model_max_length = 10**9

ck = torch.load("ablation_best.pt", map_location="cuda")
cfg = ck["config"]
abl = GPT(cfg["vocab_size"], cfg["n_embd"], cfg["block_size"], cfg["n_head"], cfg["n_layer"]).cuda()
abl.load_state_dict(ck["model"]); abl.eval()

@torch.no_grad()
def bpb(model, ctx=512):
    ids = torch.tensor(tok(TEXT, add_special_tokens=False)["input_ids"], dtype=torch.long)
    n = len(ids)//ctx; C = ids[:n*ctx].view(n, ctx)
    nll = 0.0
    for i in range(0, len(C), 4):
        x = C[i:i+4].cuda(); lo = model(x)
        lo = lo[0] if isinstance(lo, tuple) else getattr(lo, "logits", lo)
        nll += torch.nn.functional.cross_entropy(
            lo[:, :-1, :].float().reshape(-1, lo.size(-1)), x[:, 1:].reshape(-1),
            reduction="sum").item()
    return (nll/math.log(2))/BYTES

b = bpb(abl)
print("="*60)
print("TOKENIZER ABLATION RESULT")
print(f"  your 103M + YOUR tokenizer      : 1.1050 bpb  (from paper)")
print(f"  {sum(p.numel() for p in abl.parameters())/1e6:.0f}M + mGPT tokenizer, same text : {b:.4f} bpb")
print("="*60)
print("If the ablation (bigger model, worse tokenizer) scores WORSE than 1.1050,")
print("the tokenizer is the lever -> you can restore 'shown' in the paper.")